# MBTI Personality Type Prediction — Fixed & Improved Pipeline

### What Was Wrong in the Original Notebook?
1. **Severe Truncation (`max_len = 150`)**: Each row in `mbti_1.csv` contains 50 posts (~1,200–1,400 words). Truncating to 150 words threw away **~88% of every person's writing**.
2. **Scratch BiLSTM Overfitting**: Training a randomly initialized 2.56M-parameter embedding + BiLSTM on only 6,940 users caused severe memorization (`train_acc -> 81%`, `val_acc stuck at 19–21%`, `val_loss -> 4.26`).
3. **Majority-Class Collapse in the Multi-Output Model**: The original multi-output BiLSTM looked good on raw accuracy (`78% I/E`, `86% N/S`, `61% J/P`, `54% T/F`), but those numbers are literally the **dataset majority-class baselines** (`INFP`). It was mostly predicting the majority class rather than learning real personality traits.
4. **Data Leakage & Forum-Jargon Cheating**:
   - The Keras `Tokenizer` was fit on the entire dataset *before* `train_test_split`.
   - Posts come from PersonalityCafe where users explicitly write their own type (e.g., `"as an INTJ..."`, `"ENTP grin"`). Keeping the 16 MBTI type names lets the model cheat during training and fail on real-world text.
5. **Unstratified Multi-Output Split & Destroyed Stylometry**: Stripping all punctuation removed rich personality cues (`!`, `?`, `...`, pronouns, capitalization), and the multi-output split lacked `stratify`.
6. **Extreme 16-Class Weights**: Balancing 16 classes directly gave rare types like `ESTJ` (~39 samples) ~47x weight, destabilizing training (`val_acc` crashed to ~8%).

### Key Improvements in This Notebook
- **Zero Truncation**: Processes **100% of all 50 posts** using Word (1–2) + Character (3–5) TF-IDF n-grams plus **16 length-invariant stylometric features**.
- **Anti-Leakage Cleaning**: Toggleable `REMOVE_MBTI_WORDS = True` strips explicit MBTI labels from text so the predictor generalizes to real-world writing, and all vectorizers/scalers are fit strictly on training data.
- **4-Dimension Balanced Ensemble + Threshold Tuning**: Decomposes MBTI into `I/E`, `N/S`, `T/F`, `J/P` using a balanced ensemble (`LogisticRegression` + balanced `Calibrated LinearSVC` + `SGDClassifier` + `LightGBM` on SVD+Stylometry) with validation-tuned blend weights and Macro-F1 thresholds.
- **Honest Metrics, Explainability & Inference**: Reports ROC-AUC, Macro-F1, Balanced Accuracy, 4-letter exact match, top-3 accuracy, top predictive words per axis, and provides a ready-to-use `MBTIPredictor` (single text, multi-post list, or interactive Q&A).

In [ ]:
import os
import re
import zipfile
import warnings
import itertools
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import sparse
from scipy.optimize import minimize

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    log_loss,
)

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    from sklearn.ensemble import HistGradientBoostingClassifier
    HAS_LGBM = False

warnings.filterwarnings("ignore")

# Reproducibility & key modeling toggle
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# IMPORTANT:
# Set True (Recommended) for predicting REAL people's MBTI from normal text
# where they don't mention "INTJ", "INFP", etc.
# Set False only if comparing against Kaggle benchmarks that keep MBTI words.
REMOVE_MBTI_WORDS = True

print(f"LightGBM available: {HAS_LGBM}")
print(f"Remove explicit MBTI words (anti-cheating mode): {REMOVE_MBTI_WORDS}")

In [ ]:
# Flexible loader: works in Google Colab (/content/mbti_1.csv.zip) or local folder
candidate_paths = [
    "/content/mbti_1.csv.zip",
    "/content/mbti_1.csv",
    "mbti_1.csv.zip",
    "mbti_1.csv",
]

df = None
for path in candidate_paths:
    if os.path.exists(path):
        if path.endswith(".zip"):
            with zipfile.ZipFile(path, "r") as zip_ref:
                csv_filename = [f for f in zip_ref.namelist() if f.endswith(".csv")][0]
                with zip_ref.open(csv_filename) as f:
                    df = pd.read_csv(f)
            print(f"Loaded '{csv_filename}' from '{path}'")
        else:
            df = pd.read_csv(path)
            print(f"Loaded '{path}'")
        break

if df is None:
    raise FileNotFoundError(
        "Could not find mbti_1.csv.zip or mbti_1.csv. "
        "Please upload it to /content/ (Colab) or place it in the current working directory."
    )

df = df.dropna(subset=["type", "posts"]).reset_index(drop=True)
df["type"] = df["type"].str.strip().str.upper()

print(f"Dataset shape: {df.shape}")
display(df.head())

### 1. Exploratory Data Analysis (EDA)
Let's inspect:
1. The **16-class imbalance** (why 16-class softmax + extreme weights broke the original model).
2. The **4-dimension imbalance** (`I/E`, `N/S`, `T/F`, `J/P`) — notice how predicting the majority class alone gives ~77% on I/E and ~86% on N/S!
3. The **word-count distribution per user** — notice how `max_len = 150` in the original notebook discarded nearly the entire document!

In [ ]:
# Create binary dimension columns (1 = I/N/T/J, 0 = E/S/F/P)
DIMENSIONS = [
    ("IE", "I", "E", "Introversion (I) vs Extraversion (E)"),
    ("NS", "N", "S", "Intuition (N) vs Sensing (S)"),
    ("TF", "T", "F", "Thinking (T) vs Feeling (F)"),
    ("JP", "J", "P", "Judging (J) vs Perceiving (P)"),
]

for col, pos_letter, neg_letter, _ in DIMENSIONS:
    idx = {"IE": 0, "NS": 1, "TF": 2, "JP": 3}[col]
    df[col] = (df["type"].str[idx] == pos_letter).astype(int)

# Approximate word count per user (all 50 posts combined)
word_counts = df["posts"].str.replace("|||", " ", regex=False).str.split().str.len()
print(
    f"Words per user — Mean: {word_counts.mean():.0f}, "
    f"Median: {word_counts.median():.0f}, "
    f"90th percentile: {word_counts.quantile(0.9):.0f}"
)
print(
    f"Fraction of text kept by original max_len=150: "
    f"{np.minimum(150, word_counts).sum() / word_counts.sum():.1%}\n"
)

fig, axes = plt.subplots(1, 3, figsize=(19, 5))

# Plot 1: 16 MBTI classes
type_counts = df["type"].value_counts()
sns.barplot(
    x=type_counts.index,
    y=type_counts.values,
    hue=type_counts.index,
    palette="viridis",
    legend=False,
    ax=axes[0],
)
axes[0].set_title("16 MBTI Classes (Severe Imbalance)")
axes[0].set_xlabel("MBTI Type")
axes[0].set_ylabel("Users")
axes[0].tick_params(axis="x", rotation=45)
axes[0].grid(axis="y", linestyle="--", alpha=0.5)

# Plot 2: 4 binary dimensions
dim_stats = []
for col, pos_l, neg_l, _ in DIMENSIONS:
    pos_pct = df[col].mean() * 100
    dim_stats.append({"Dimension": f"{pos_l}/{neg_l}", "Majority %": max(pos_pct, 100 - pos_pct), "Positive %": pos_pct})
dim_df = pd.DataFrame(dim_stats)
sns.barplot(x="Dimension", y="Positive %", data=dim_df, palette="coolwarm", hue="Dimension", legend=False, ax=axes[1])
axes[1].axhline(50, color="black", linestyle="--", linewidth=1, label="Balanced (50%)")
axes[1].set_ylim(0, 100)
axes[1].set_title("% of First Letter (I, N, T, J) per Axis")
axes[1].set_ylabel("Percentage (%)")
axes[1].legend()
axes[1].grid(axis="y", linestyle="--", alpha=0.5)

# Plot 3: Word count distribution vs original max_len=150
axes[2].hist(word_counts, bins=40, color="#3b82f6", alpha=0.8, edgecolor="white")
axes[2].axvline(150, color="red", linestyle="--", linewidth=2.2, label="Old max_len = 150")
axes[2].set_title("Total Words per User (Original Cutoff in Red)")
axes[2].set_xlabel("Word Count")
axes[2].set_ylabel("Users")
axes[2].legend()
axes[2].grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

### 2. Leak-Free Text Cleaning & Length-Invariant Stylometric Features
Two critical design choices make this predictor work on **real human writing** rather than just forum benchmarks:
1. **Explicit MBTI Word Removal**: Regex strips all 16 type names (`infj`, `intp`, `entps`, `mbti`, etc.) when `REMOVE_MBTI_WORDS = True`.
2. **Length-Invariant Stylometric Features**: Instead of raw counts (which fail when predicting on a 200-word sample vs. a 1,300-word training profile), every stylometric feature is normalized as a **rate per 100 words** or a **ratio** — including punctuation (`!`, `?`, `...`), capitalization, emoticons, pronoun balance (`I` vs `we` vs `you`), negation, analytical vs. feeling vocabulary, and moving-average lexical diversity (MATTR).

In [ ]:
MBTI_TYPES = [
    "infj", "infp", "intj", "intp",
    "isfj", "isfp", "istj", "istp",
    "enfj", "enfp", "entj", "entp",
    "esfj", "esfp", "estj", "estp",
]
MBTI_PATTERN = re.compile(
    r"\b(?:" + "|".join(MBTI_TYPES) + r")s?\b|\bmbti\b|\bpersonalitycafe\b",
    flags=re.IGNORECASE,
)
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
EMOTICON_PATTERN = re.compile(
    r"(?::|;|=)(?:-)?(?:\)|\(|D|P|p|/|\\|O|o|\*)|<3|\b[xX]D\b|\^_\^"
)

FIRST_PERSON_SG = {"i", "me", "my", "mine", "myself", "im", "ive", "id"}
FIRST_PERSON_PL = {"we", "us", "our", "ours", "ourselves", "were", "weve"}
SECOND_PERSON = {"you", "your", "yours", "yourself", "yourselves", "youre", "youve"}
THIRD_PERSON = {"he", "him", "his", "she", "her", "hers", "they", "them", "their", "theirs"}
NEGATIONS = {"not", "no", "never", "none", "nothing", "neither", "nor", "dont", "cant", "wont", "isnt", "arent", "wasnt", "werent", "didnt"}
ANALYTICAL_WORDS = {
    "think", "thought", "logic", "logical", "reason", "rational", "analyze", "analysis",
    "theory", "system", "objective", "data", "fact", "facts", "evidence", "efficient",
    "because", "therefore", "however", "actually", "concept", "argument", "debate", "strategy",
}
FEELING_WORDS = {
    "feel", "feeling", "feelings", "love", "heart", "care", "caring", "empathy", "compassion",
    "happy", "sad", "hurt", "hope", "sorry", "warm", "kind", "beautiful", "emotional",
    "harmony", "values", "passion", "grateful", "appreciate", "relationship",
}
PLANNING_WORDS = {
    "plan", "plans", "schedule", "organize", "organized", "structure", "deadline", "goal",
    "goals", "finish", "complete", "list", "order", "routine", "decide", "decided", "always",
}
SPONTANEOUS_WORDS = {
    "maybe", "might", "whatever", "spontaneous", "random", "flexible", "explore", "options",
    "depends", "later", "procrastinate", "sometimes", "guess", "wonder", "improvise", "curious",
}


class TextCleaner(BaseEstimator, TransformerMixin):
    """Cleans raw text while optionally stripping explicit MBTI type mentions."""

    def __init__(self, remove_mbti_words=True):
        self.remove_mbti_words = remove_mbti_words

    def fit(self, X, y=None):
        return self

    def _clean_one(self, text):
        text = str(text)
        text = URL_PATTERN.sub(" ", text)
        text = text.replace("|||", " ")
        text = text.lower()
        # Normalize common contractions by removing apostrophes inside words (e.g. don't -> dont)
        text = re.sub(r"['’]", "", text)
        if self.remove_mbti_words:
            text = MBTI_PATTERN.sub(" ", text)
        # Keep alphabetic characters and whitespace
        text = re.sub(r"[^a-z\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def transform(self, X):
        return [self._clean_one(t) for t in X]


class StylometricExtractor(BaseEstimator, TransformerMixin):
    """Extracts 16 length-invariant linguistic & punctuation features from raw text."""

    FEATURE_NAMES = [
        "exclam_per_100w",
        "question_per_100w",
        "ellipsis_per_100w",
        "allcaps_per_100w",
        "emoticon_per_100w",
        "url_per_100w",
        "avg_word_len",
        "words_per_sentence",
        "mattr_lexical_div",
        "first_sg_per_100w",
        "first_pl_per_100w",
        "second_p_per_100w",
        "third_p_per_100w",
        "negation_per_100w",
        "analytical_vs_feeling",
        "planning_vs_spontaneous",
    ]

    def fit(self, X, y=None):
        return self

    def _extract_one(self, raw_text):
        raw = str(raw_text)
        n_urls = len(URL_PATTERN.findall(raw))
        n_emoticons = len(EMOTICON_PATTERN.findall(raw))
        no_url = URL_PATTERN.sub(" ", raw)

        n_exclam = no_url.count("!")
        n_question = no_url.count("?")
        n_ellipsis = len(re.findall(r"\.{3,}", no_url))

        # Sentence/post segments
        segments = [s.strip() for s in re.split(r"\|\|\||[.!?]+", no_url) if s.strip()]
        n_segments = max(len(segments), 1)

        raw_words = re.findall(r"[A-Za-z]+", no_url)
        n_words = max(len(raw_words), 1)
        scale = 100.0 / n_words

        n_allcaps = sum(1 for w in raw_words if len(w) >= 2 and w.isupper())
        avg_word_len = sum(len(w) for w in raw_words) / n_words
        words_per_sent = n_words / n_segments

        # Normalize words for lexicon lookups
        norm_text = re.sub(r"['’]", "", no_url.lower())
        words = re.findall(r"[a-z]+", norm_text)
        n_norm = max(len(words), 1)
        norm_scale = 100.0 / n_norm

        # Length-invariant lexical diversity (Moving-Average TTR over 50-word windows)
        win = 50
        if len(words) <= win:
            mattr = len(set(words)) / n_norm
        else:
            step = max(1, (len(words) - win) // 20)
            ttrs = [len(set(words[i : i + win])) / win for i in range(0, len(words) - win + 1, step)]
            mattr = float(np.mean(ttrs))

        c_1sg = sum(1 for w in words if w in FIRST_PERSON_SG)
        c_1pl = sum(1 for w in words if w in FIRST_PERSON_PL)
        c_2p = sum(1 for w in words if w in SECOND_PERSON)
        c_3p = sum(1 for w in words if w in THIRD_PERSON)
        c_neg = sum(1 for w in words if w in NEGATIONS)
        c_ana = sum(1 for w in words if w in ANALYTICAL_WORDS)
        c_feel = sum(1 for w in words if w in FEELING_WORDS)
        c_plan = sum(1 for w in words if w in PLANNING_WORDS)
        c_spon = sum(1 for w in words if w in SPONTANEOUS_WORDS)

        ana_vs_feel = (c_ana - c_feel) / (c_ana + c_feel + 1.0)
        plan_vs_spon = (c_plan - c_spon) / (c_plan + c_spon + 1.0)

        return [
            n_exclam * scale,
            n_question * scale,
            n_ellipsis * scale,
            n_allcaps * scale,
            n_emoticons * scale,
            n_urls * scale,
            avg_word_len,
            min(words_per_sent, 60.0),
            mattr,
            c_1sg * norm_scale,
            c_1pl * norm_scale,
            c_2p * norm_scale,
            c_3p * norm_scale,
            c_neg * norm_scale,
            ana_vs_feel,
            plan_vs_spon,
        ]

    def transform(self, X):
        return np.array([self._extract_one(t) for t in X], dtype=np.float32)


class MBTIFeatureBuilder:
    """Builds leak-free sparse (TF-IDF + normalized stylometry) and dense (SVD + stylometry) features."""

    def __init__(self, remove_mbti_words=True):
        self.cleaner = TextCleaner(remove_mbti_words=remove_mbti_words)
        self.word_tfidf = TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=3,
            max_df=0.90,
            max_features=35000,
            sublinear_tf=True,
            strip_accents="unicode",
        )
        self.char_tfidf = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=5,
            max_df=0.90,
            max_features=25000,
            sublinear_tf=True,
        )
        self.stylo = StylometricExtractor()
        self.stylo_scaler = StandardScaler()
        self.stylo_norm = Normalizer(norm="l2")
        self.svd = TruncatedSVD(n_components=120, random_state=RANDOM_STATE)

    def fit_transform(self, raw_texts):
        clean_texts = self.cleaner.fit_transform(raw_texts)
        X_word = self.word_tfidf.fit_transform(clean_texts)
        X_char = self.char_tfidf.fit_transform(clean_texts)

        X_stylo_raw = self.stylo.fit_transform(raw_texts)
        X_stylo_scaled = self.stylo_scaler.fit_transform(X_stylo_raw)
        # Scale L2-normalized stylometric block mildly so TF-IDF and stylometry stay well-proportioned
        X_stylo_sparse = sparse.csr_matrix(self.stylo_norm.fit_transform(X_stylo_scaled) * 0.35)

        X_tfidf = sparse.hstack([X_word, X_char], format="csr")
        X_svd = self.svd.fit_transform(X_tfidf).astype(np.float32)

        X_sparse = sparse.hstack([X_word, X_char, X_stylo_sparse], format="csr")
        X_dense = np.hstack([X_svd, X_stylo_scaled]).astype(np.float32)
        return X_sparse, X_dense

    def transform(self, raw_texts):
        clean_texts = self.cleaner.transform(raw_texts)
        X_word = self.word_tfidf.transform(clean_texts)
        X_char = self.char_tfidf.transform(clean_texts)

        X_stylo_raw = self.stylo.transform(raw_texts)
        X_stylo_scaled = self.stylo_scaler.transform(X_stylo_raw)
        X_stylo_sparse = sparse.csr_matrix(self.stylo_norm.transform(X_stylo_scaled) * 0.35)

        X_tfidf = sparse.hstack([X_word, X_char], format="csr")
        X_svd = self.svd.transform(X_tfidf).astype(np.float32)

        X_sparse = sparse.hstack([X_word, X_char, X_stylo_sparse], format="csr")
        X_dense = np.hstack([X_svd, X_stylo_scaled]).astype(np.float32)
        return X_sparse, X_dense

print("Feature classes initialized successfully.")

### 3. Stratified Train / Validation / Test Split (Zero Leakage)
We split the dataset **stratified by all 16 MBTI types**:
- **70% Train**: Used to fit TF-IDF, SVD, scalers, and base models.
- **15% Validation**: Used *only* to learn ensemble weights and optimal decision thresholds per dimension.
- **15% Test**: Completely untouched until final evaluation!

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=RANDOM_STATE, stratify=df["type"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=RANDOM_STATE, stratify=temp_df["type"]
)

print(f"Train: {len(train_df)} | Validation: {len(val_df)} | Test: {len(test_df)}")

# Fit feature pipeline STRICTLY on training texts
feat_builder = MBTIFeatureBuilder(remove_mbti_words=REMOVE_MBTI_WORDS)
X_train_sp, X_train_dn = feat_builder.fit_transform(train_df["posts"])
X_val_sp, X_val_dn = feat_builder.transform(val_df["posts"])
X_test_sp, X_test_dn = feat_builder.transform(test_df["posts"])

print(f"Sparse feature matrix shape (Word + Char TF-IDF + Stylometry): {X_train_sp.shape}")
print(f"Dense feature matrix shape  (SVD-120 + Stylometry):            {X_train_dn.shape}")

### 4. Balanced 4-Dimension Ensemble & Threshold Optimization
For each MBTI dimension (`I/E`, `N/S`, `T/F`, `J/P`), we train 4 diverse **class-balanced** models so every axis learns a fair 50/50 boundary (instead of collapsing to the forum majority `INFP`):
1. **L2 Logistic Regression** (sparse TF-IDF + stylometry)
2. **Balanced Calibrated LinearSVC** (sparse TF-IDF + stylometry)
3. **Log-Loss SGD Classifier** (sparse TF-IDF + stylometry)
4. **LightGBM** (dense TruncatedSVD + stylometric interactions)

Then we optimize the **ensemble weights** (minimizing balanced log-loss on Validation) and pick the **optimal probability threshold** (maximizing Macro F1 on Validation).

In [ ]:
def make_gbdt():
    if HAS_LGBM:
        return LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1,
        )
    return HistGradientBoostingClassifier(
        max_iter=250,
        learning_rate=0.05,
        max_leaf_nodes=31,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )


def fit_axis_ensemble(X_tr_sp, X_tr_dn, y_tr, X_va_sp=None, X_va_dn=None, y_va=None, fixed_weights=None, fixed_threshold=None):
    """Trains 4 balanced models for one MBTI dimension and optionally tunes weights & threshold on Val."""
    sw_tr = compute_sample_weight("balanced", y_tr)

    # 1. Regularized Logistic Regression
    lr = LogisticRegression(
        C=1.5, class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE
    )
    lr.fit(X_tr_sp, y_tr)

    # 2. Balanced Calibrated LinearSVC (balanced once via sample_weight so calibrator stays 50/50)
    cal_svc = CalibratedClassifierCV(
        estimator=LinearSVC(C=0.25, max_iter=3000, random_state=RANDOM_STATE),
        method="sigmoid",
        cv=3,
    )
    cal_svc.fit(X_tr_sp, y_tr, sample_weight=sw_tr)

    # 3. Regularized SGD (log-loss)
    sgd = SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=1e-4,
        class_weight="balanced",
        max_iter=1000,
        tol=1e-4,
        random_state=RANDOM_STATE,
    )
    sgd.fit(X_tr_sp, y_tr)

    # 4. Gradient Boosted Trees on dense SVD + stylometry
    gbdt = make_gbdt()
    gbdt.fit(X_tr_dn, y_tr)

    models = {"lr": lr, "svc": cal_svc, "sgd": sgd, "gbdt": gbdt}

    if fixed_weights is not None and fixed_threshold is not None:
        return {"models": models, "weights": fixed_weights, "threshold": fixed_threshold}

    # Predict probabilities on validation set
    p_val_mat = np.column_stack([
        lr.predict_proba(X_va_sp)[:, 1],
        cal_svc.predict_proba(X_va_sp)[:, 1],
        sgd.predict_proba(X_va_sp)[:, 1],
        gbdt.predict_proba(X_va_dn)[:, 1],
    ])

    sw_va = compute_sample_weight("balanced", y_va)

    def obj(w):
        w = np.clip(w, 0, 1)
        w = w / (w.sum() + 1e-12)
        p_blend = np.clip(p_val_mat @ w, 1e-6, 1 - 1e-6)
        return log_loss(y_va, p_blend, sample_weight=sw_va)

    res = minimize(
        obj,
        x0=np.array([0.40, 0.30, 0.15, 0.15]),
        bounds=[(0.0, 1.0)] * 4,
        constraints={"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
        method="SLSQP",
    )
    best_weights = res.x / res.x.sum()
    p_val_blend = p_val_mat @ best_weights

    # Tune decision threshold on validation set to maximize Macro F1
    best_thr, best_f1 = 0.50, -1.0
    for thr in np.linspace(0.30, 0.70, 81):
        score = f1_score(y_va, (p_val_blend >= thr).astype(int), average="macro")
        if score > best_f1:
            best_f1 = score
            best_thr = float(thr)

    return {
        "models": models,
        "weights": best_weights,
        "threshold": best_thr,
        "val_auc": roc_auc_score(y_va, p_val_blend),
        "val_macro_f1": best_f1,
    }


def predict_axis_proba(axis_bundle, X_sp, X_dn):
    m = axis_bundle["models"]
    w = axis_bundle["weights"]
    p_mat = np.column_stack([
        m["lr"].predict_proba(X_sp)[:, 1],
        m["svc"].predict_proba(X_sp)[:, 1],
        m["sgd"].predict_proba(X_sp)[:, 1],
        m["gbdt"].predict_proba(X_dn)[:, 1],
    ])
    return p_mat @ w


def align_proba_to_threshold(p, thr):
    """Smoothly maps raw probability p so that p == thr maps to 0.50 (preserves ROC-AUC)."""
    p = np.asarray(p, dtype=float)
    return np.where(
        p >= thr,
        0.5 + 0.5 * (p - thr) / max(1.0 - thr, 1e-6),
        0.5 * p / max(thr, 1e-6),
    )


axis_bundles = {}
for col, pos_l, neg_l, desc in DIMENSIONS:
    print(f"Training ensemble for {desc}...")
    bundle = fit_axis_ensemble(
        X_train_sp, X_train_dn, train_df[col].values,
        X_val_sp, X_val_dn, val_df[col].values,
    )
    axis_bundles[col] = bundle
    w_str = ", ".join(f"{k}:{v:.2f}" for k, v in zip(["LR", "SVC", "SGD", "GBDT"], bundle["weights"]))
    print(
        f"  -> Val ROC-AUC: {bundle['val_auc']:.4f} | "
        f"Val Macro-F1: {bundle['val_macro_f1']:.4f} | "
        f"Threshold: {bundle['threshold']:.3f} | Weights: [{w_str}]"
    )

### 5. Honest Evaluation on Untouched Test Set
Now let's evaluate on the completely held-out **Test** set and check:
- **ROC-AUC, Macro-F1, and Balanced Accuracy** per axis (unlike raw accuracy, these cannot be gamed by predicting the majority class!).
- **Exact 4-letter MBTI Match Accuracy**, **Average Letters Correct (out of 4)**, and **Top-3 MBTI Accuracy**.
- **ROC Curves & Confusion Matrices** for all 4 dimensions.

In [ ]:
results = []
test_probs_raw = {}
test_probs_aligned = {}
test_preds = {}

for col, pos_l, neg_l, desc in DIMENSIONS:
    y_true = test_df[col].values
    p_raw = predict_axis_proba(axis_bundles[col], X_test_sp, X_test_dn)
    thr = axis_bundles[col]["threshold"]
    y_pred = (p_raw >= thr).astype(int)

    test_probs_raw[col] = p_raw
    test_probs_aligned[col] = align_proba_to_threshold(p_raw, thr)
    test_preds[col] = y_pred

    maj_acc = max(y_true.mean(), 1.0 - y_true.mean())
    results.append({
        "Dimension": f"{pos_l}/{neg_l}",
        "ROC-AUC": roc_auc_score(y_true, p_raw),
        "Macro F1": f1_score(y_true, y_pred, average="macro"),
        "Balanced Acc": balanced_accuracy_score(y_true, y_pred),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Majority Baseline Acc": maj_acc,
    })

results_df = pd.DataFrame(results)
display(results_df.style.format({
    "ROC-AUC": "{:.4f}",
    "Macro F1": "{:.4f}",
    "Balanced Acc": "{:.4f}",
    "Accuracy": "{:.4f}",
    "Majority Baseline Acc": "{:.4f}",
}))

# Reconstruct 4-letter MBTI predictions & compute joint probabilities over all 16 types
pred_types = []
top3_hits = 0
letters_correct = []

all_16_combos = list(itertools.product([("I", 1), ("E", 0)], [("N", 1), ("S", 0)], [("T", 1), ("F", 0)], [("J", 1), ("P", 0)]))

for i in range(len(test_df)):
    true_type = test_df["type"].iloc[i]
    p_i = test_probs_aligned["IE"][i]
    p_n = test_probs_aligned["NS"][i]
    p_t = test_probs_aligned["TF"][i]
    p_j = test_probs_aligned["JP"][i]

    type_scores = []
    for (l0, b0), (l1, b1), (l2, b2), (l3, b3) in all_16_combos:
        prob = (
            (p_i if b0 else 1 - p_i)
            * (p_n if b1 else 1 - p_n)
            * (p_t if b2 else 1 - p_t)
            * (p_j if b3 else 1 - p_j)
        )
        type_scores.append((l0 + l1 + l2 + l3, prob))

    type_scores.sort(key=lambda x: x[1], reverse=True)
    top1 = type_scores[0][0]
    top3 = [t for t, _ in type_scores[:3]]

    pred_types.append(top1)
    if true_type in top3:
        top3_hits += 1
    letters_correct.append(sum(a == b for a, b in zip(top1, true_type)))

exact_acc = np.mean([p == t for p, t in zip(pred_types, test_df["type"])])
top3_acc = top3_hits / len(test_df)
avg_letters = np.mean(letters_correct)
at_least_3 = np.mean([c >= 3 for c in letters_correct])

print("\n" + "=" * 60)
print(f"OVERALL TEST PERFORMANCE (REMOVE_MBTI_WORDS = {REMOVE_MBTI_WORDS})")
print("=" * 60)
print(f"Exact 4-Letter MBTI Accuracy: {exact_acc:.2%}  (Original BiLSTM: ~19–21%)")
print(f"Top-3 MBTI Type Accuracy:     {top3_acc:.2%}")
print(f"3+ out of 4 Letters Correct:  {at_least_3:.2%}")
print(f"Average Letters Correct:      {avg_letters:.2f} / 4.00")
print("=" * 60)

# Plot ROC Curves & Confusion Matrices
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for idx, (col, pos_l, neg_l, desc) in enumerate(DIMENSIONS):
    y_true = test_df[col].values
    p_raw = test_probs_raw[col]
    y_pred = test_preds[col]

    # Row 0: ROC Curve
    fpr, tpr, _ = roc_curve(y_true, p_raw)
    auc_val = roc_auc_score(y_true, p_raw)
    axes[0, idx].plot(fpr, tpr, color="#2563eb", lw=2, label=f"AUC = {auc_val:.3f}")
    axes[0, idx].plot([0, 1], [0, 1], color="gray", linestyle="--", lw=1)
    axes[0, idx].set_title(f"ROC: {pos_l} vs {neg_l}")
    axes[0, idx].set_xlabel("False Positive Rate")
    axes[0, idx].set_ylabel("True Positive Rate")
    axes[0, idx].legend(loc="lower right")
    axes[0, idx].grid(alpha=0.4)

    # Row 1: Normalized Confusion Matrix
    cm = confusion_matrix(y_true, y_pred, normalize="true")
    sns.heatmap(
        cm,
        annot=True,
        fmt=".2%",
        cmap="Blues",
        xticklabels=[f"Pred {neg_l}", f"Pred {pos_l}"],
        yticklabels=[f"True {neg_l}", f"True {pos_l}"],
        cbar=False,
        ax=axes[1, idx],
    )
    axes[1, idx].set_title(f"Confusion Matrix: {pos_l}/{neg_l}")

plt.tight_layout()
plt.show()

### 6. Model Interpretability — What Words Drive Each Personality Axis?
Checking the top weighted words from our Logistic Regression branch verifies that the model is learning genuine linguistic patterns (and **not** cheating on forum type labels).

In [ ]:
word_vocab = feat_builder.word_tfidf.get_feature_names_out()
n_word_feats = len(word_vocab)

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
axes = axes.ravel()

for idx, (col, pos_l, neg_l, desc) in enumerate(DIMENSIONS):
    coefs = axis_bundles[col]["models"]["lr"].coef_[0][:n_word_feats]
    top_pos_idx = np.argsort(coefs)[-10:]
    top_neg_idx = np.argsort(coefs)[:10]

    sel_idx = np.concatenate([top_neg_idx, top_pos_idx])
    sel_words = word_vocab[sel_idx]
    sel_vals = coefs[sel_idx]
    colors = ["#ef4444" if v < 0 else "#2563eb" for v in sel_vals]

    axes[idx].barh(sel_words, sel_vals, color=colors)
    axes[idx].axvline(0, color="black", linewidth=0.9)
    axes[idx].set_title(f"{desc}\n(Red -> {neg_l} | Blue -> {pos_l})")
    axes[idx].set_xlabel("Logistic Regression Weight")
    axes[idx].grid(axis="x", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

### 7. Refit Final Production Predictor & Save to Disk
Now that we have verified leak-free performance on the untouched test set, we refit the feature pipeline and 4-axis ensembles on **100% of the available data** (using the validation-tuned blend weights and thresholds) and package everything into a clean `MBTIPredictor` object saved to `mbti_predictor.joblib`.

In [ ]:
MBTI_DESCRIPTIONS = {
    "INTJ": "Architect — Strategic, independent, analytical planner",
    "INTP": "Logician — Curious, theoretical, inventive problem-solver",
    "ENTJ": "Commander — Bold, decisive, goal-driven organizer",
    "ENTP": "Debater — Quick-witted, playful, idea-exploring challenger",
    "INFJ": "Advocate — Insightful, idealistic, empathetic visionary",
    "INFP": "Mediator — Imaginative, values-driven, introspective idealist",
    "ENFJ": "Protagonist — Charismatic, warm, people-centered leader",
    "ENFP": "Campaigner — Enthusiastic, creative, free-spirited connector",
    "ISTJ": "Logistician — Practical, thorough, dependable organizer",
    "ISFJ": "Defender — Supportive, attentive, loyal protector",
    "ESTJ": "Executive — Direct, structured, results-oriented manager",
    "ESFJ": "Consul — Sociable, caring, community-minded helper",
    "ISTP": "Virtuoso — Hands-on, adaptable, calm troubleshooter",
    "ISFP": "Adventurer — Artistic, gentle, present-focused explorer",
    "ESTP": "Entrepreneur — Energetic, pragmatic, action-oriented doer",
    "ESFP": "Entertainer — Spontaneous, lively, expressive optimist",
}


class MBTIPredictor:
    """End-to-end MBTI predictor with calibrated axis probabilities and top-K ranking."""

    def __init__(self, feature_builder, axis_bundles):
        self.feature_builder = feature_builder
        self.axis_bundles = axis_bundles
        self.all_16_combos = list(itertools.product(
            [("I", 1), ("E", 0)],
            [("N", 1), ("S", 0)],
            [("T", 1), ("F", 0)],
            [("J", 1), ("P", 0)],
        ))

    def predict(self, text_or_posts):
        if isinstance(text_or_posts, (list, tuple)):
            combined_text = " ||| ".join(str(p).strip() for p in text_or_posts if str(p).strip())
        else:
            combined_text = str(text_or_posts).strip()

        word_count = len(re.findall(r"[A-Za-z]+", combined_text))
        X_sp, X_dn = self.feature_builder.transform([combined_text])

        axis_out = {}
        aligned_probs = {}
        for col, pos_l, neg_l, desc in DIMENSIONS:
            bundle = self.axis_bundles[col]
            p_raw = float(predict_axis_proba(bundle, X_sp, X_dn)[0])
            p_adj = float(align_proba_to_threshold(p_raw, bundle["threshold"]))
            aligned_probs[col] = p_adj
            axis_out[col] = {
                "predicted": pos_l if p_adj >= 0.5 else neg_l,
                pos_l: p_adj,
                neg_l: 1.0 - p_adj,
                "label": desc,
            }

        # Joint probability over all 16 types
        p_i, p_n, p_t, p_j = (
            aligned_probs["IE"],
            aligned_probs["NS"],
            aligned_probs["TF"],
            aligned_probs["JP"],
        )
        type_scores = []
        for (l0, b0), (l1, b1), (l2, b2), (l3, b3) in self.all_16_combos:
            prob = (
                (p_i if b0 else 1 - p_i)
                * (p_n if b1 else 1 - p_n)
                * (p_t if b2 else 1 - p_t)
                * (p_j if b3 else 1 - p_j)
            )
            type_scores.append((l0 + l1 + l2 + l3, prob))
        type_scores.sort(key=lambda x: x[1], reverse=True)

        predicted_type = type_scores[0][0]
        return {
            "predicted_type": predicted_type,
            "description": MBTI_DESCRIPTIONS.get(predicted_type, ""),
            "word_count": word_count,
            "reliability": (
                "High (200+ words)" if word_count >= 200
                else "Moderate (80–199 words)" if word_count >= 80
                else "Low (< 80 words — write more for a reliable prediction!)"
            ),
            "axes": axis_out,
            "top_3_types": type_scores[:3],
        }

    def explain_prediction(self, text_or_posts):
        res = self.predict(text_or_posts)
        print("=" * 64)
        print(f"  Predicted MBTI Type: {res['predicted_type']}  —  {res['description']}")
        print(f"  Input Length:        {res['word_count']} words  |  Reliability: {res['reliability']}")
        print("-" * 64)
        for col, pos_l, neg_l, _ in DIMENSIONS:
            ax = res["axes"][col]
            p_pos = ax[pos_l]
            bar_len = 24
            filled = int(round(p_pos * bar_len))
            bar = "█" * filled + "░" * (bar_len - filled)
            print(f"  {pos_l} {p_pos:5.1%} [{bar}] {ax[neg_l]:5.1%} {neg_l}   -> {ax['predicted']}")
        print("-" * 64)
        top3_str = "  |  ".join(f"{t} ({p:.1%})" for t, p in res["top_3_types"])
        print(f"  Top 3 Likely Types: {top3_str}")
        print("=" * 64)
        return res


# Refit on 100% of data using the validation-tuned weights and thresholds
print("Refitting final production model on full dataset...")
full_feat_builder = MBTIFeatureBuilder(remove_mbti_words=REMOVE_MBTI_WORDS)
X_full_sp, X_full_dn = full_feat_builder.fit_transform(df["posts"])

full_axis_bundles = {}
for col, pos_l, neg_l, desc in DIMENSIONS:
    full_axis_bundles[col] = fit_axis_ensemble(
        X_full_sp,
        X_full_dn,
        df[col].values,
        fixed_weights=axis_bundles[col]["weights"],
        fixed_threshold=axis_bundles[col]["threshold"],
    )

predictor = MBTIPredictor(full_feat_builder, full_axis_bundles)
joblib.dump(predictor, "mbti_predictor.joblib", compress=3)
print("Saved production predictor to 'mbti_predictor.joblib'!")

### 8. Predict a Person's MBTI Type!
**Tip for the Most Accurate Prediction:**
Personality signals emerge across multiple contexts — a single 10-word sentence rarely contains enough signal for any model. Pass **150–400+ words** (such as a few journal entries, social posts, emails, or answers to the 6-question interview below) for the most reliable MBTI prediction.

In [ ]:
# --- Example 1: Analytical, structured, introspective writer ---
sample_1 = [
    "I usually spend my weekends quietly working through a long-term side project or reading books on systems design and philosophy. Large noisy parties drain my energy very quickly.",
    "Before starting any major task, I map out the entire architecture, list every milestone, and set strict deadlines so nothing is left to last-minute guesswork.",
    "When a friend asks for advice on a hard decision, I naturally analyze the logical trade-offs and objective long-term consequences rather than focusing purely on how everyone feels in the moment.",
    "I love uncovering abstract patterns and theoretical frameworks that explain why complex systems behave the way they do, rather than memorizing routine step-by-step procedures.",
]

# --- Example 2: Warm, energetic, spontaneous, people-focused writer ---
sample_2 = [
    "OMG I had the BEST weekend!! My friends and I spontaneously decided to drive out to the coast with zero itinerary and just explore random little cafes and talk to everyone we met!",
    "I get so energized bouncing wild ideas around with people! Whenever someone is having a rough day, I just want to listen with my whole heart and help them feel truly seen and cared for.",
    "Honestly I hate strict schedules and rigid routines — life is way too short not to stay open to surprises, creative inspiration, and new adventures as they pop up!",
    "I love imagining all the meaningful possibilities for the future and connecting deeply with people's stories, dreams, and values :)",
]

print("\n--- SAMPLE 1 ---")
predictor.explain_prediction(sample_1)

print("\n--- SAMPLE 2 ---")
predictor.explain_prediction(sample_2)


# --- Interactive Multi-Question MBTI Assessment ---
def run_mbti_interview(predictor_obj):
    """Collects 6 natural writing samples (~150-300 words) from a user and predicts their MBTI type."""
    prompts = [
        "1. Describe your ideal weekend from morning to night:",
        "2. How do you usually approach a big project or deadline?",
        "3. When making a tough life decision, what matters most to you?",
        "4. How do you recharge after a stressful week?",
        "5. What topics, hobbies, or conversations excite you the most?",
        "6. How do you handle conflict or disagreement with someone close to you?",
    ]
    print("Answer each question naturally in 2–4 sentences (write in your normal voice!):\n")
    answers = []
    for q in prompts:
        print(q)
        ans = input("> ").strip()
        if ans:
            answers.append(ans)
        print()
    if not answers:
        print("No answers provided.")
        return None
    return predictor_obj.explain_prediction(answers)

# Uncomment the line below to run the interactive personality interview in Colab/Jupyter!
# run_mbti_interview(predictor)